# Phase 4: Exploratory Data Analysis & Student Early Warning Signal Discovery

**Project:** Smart Student Retention & Early Warning System  
**Dataset:** Open University Learning Analytics Dataset (OULAD)  
**Cohort Definition:** Policy C (Active at Start, Survived to Cutoff Day 35)  
**Target:** `withdrawn_after_cutoff` (Withdrawal occurring strictly after relative Day 35)  

---

## Critical Methodological Framing: Association vs. Causation

> **Important Analytical Caveat:** All empirical patterns, group differences, effect sizes, and odds ratios documented in this notebook represent **observational associations**, not causal mechanisms. For example, while low VLE click count is strongly associated with course withdrawal, artificially requiring disengaged students to click links without instructional support cannot be assumed to improve retention. These features serve as operational risk markers to trigger proactive, supportive human advisor interventions.


In [ ]:
from pathlib import Path

import pandas as pd

try:
    from IPython.display import Image, display
except ImportError:
    def display(x):
        print(x)
    class Image:
        def __init__(self, filename):
            self.filename = filename
        def __repr__(self):
            return f"<Image: {self.filename}>"


def find_project_root() -> Path:
    """Locate project root by searching upwards for pyproject.toml."""
    current = Path.cwd().resolve()
    for directory in [current, *current.parents]:
        if (directory / "pyproject.toml").is_file():
            return directory
    return current.parent if (current.parent / "pyproject.toml").is_file() else current


# Configure project paths
PROJECT_ROOT = find_project_root()
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
REPORTS_DIR = PROJECT_ROOT / "outputs" / "reports"

print(f"Project Root: {PROJECT_ROOT}")
print(f"Tables Directory: {TABLES_DIR}")
print(f"Figures Directory: {FIGURES_DIR}")


## 1. Cohort Profile & Eligibility Reconciliation

Under **Policy C**, the target population consists of students who officially commenced the course and did not withdraw before the prediction cutoff (relative Day 35). This excludes 5,465 premature dropouts who withdrew before instruction meaningfully commenced or prior to Day 35.


In [ ]:
# Load Policy Comparison & Cohort Table
cohort_df = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "cohort" / "cohort_policy_c.parquet")
total_records = len(cohort_df)
positive_records = int((cohort_df["withdrawn_after_cutoff"] == 1).sum())
base_rate = positive_records / total_records * 100.0

print(f"Eligible Cohort Size (N): {total_records:,}")
print(f"Withdrawn After Day 35: {positive_records:,} ({base_rate:.2f}%)")
print(f"Retained (Completers): {total_records - positive_records:,}")


## 2. Observation 1: Severe Cross-Presentation and Faculty Domain Heterogeneity

Withdrawal rates vary substantially across the 22 module-presentations, ranging from **3.7%** in `GGG 2013J` to **28.3%** in `CCC 2014B` (mean across offerings: 16.4%). Key structural patterns emerge:
1. **Module & Discipline Heterogeneity:** In the OULAD dataset specification (Kuzilek et al., 2017), modules are categorized into STEM (`CCC`, `DDD`, `EEE`, `FFF`) and Social Sciences (`AAA`, `BBB`, `GGG`). While STEM offerings show a higher overall mean attrition rate (19.7%) than Social Sciences (11.6%), substantial intra-domain variation exists across specific modules (e.g., `EEE` has 11.9%–13.3% attrition while `CCC` has 26.0%–28.3%).
2. **Seasonality Patterns:** February (`B`) presentations exhibit higher attrition on average (18.3%) than October (`J`) presentations (15.1%) across offerings (student-weighted pooled rates are 18.9% and 16.4%, respectively). However, seasonal disparities are not uniform across all offerings: while `CCC`, `DDD`, and `EEE` experience higher attrition in February cohorts, `BBB 2014B` (11.9%), `FFF 2013B` (16.6%), and `GGG 2014B` (7.7%) had lower withdrawal rates than their respective October cohorts (`BBB 2014J`: 12.8%; `FFF 2013J`: 18.2%; `GGG 2014J`: 12.5%).

In [ ]:
pres_df = pd.read_csv(TABLES_DIR / "presentation_withdrawal_rates.csv")
display(pres_df.sort_values("withdrawal_rate_pct", ascending=False).head(10))

# Display Figure 1
display(Image(filename=str(FIGURES_DIR / "fig1_withdrawal_rate_by_presentation.png")))


## 3. Observation 2: Early Intensity and Diversity of VLE Digital Engagement

In distance-learning environments, virtual interaction volume and pacing serve as critical proxies for active student participation. Examining engagement logged strictly up to Day 35:
- **Volume Gap:** Retained students logged a mean of **430.2 clicks** (median: 275.0) across **17.7 active days** (median: 16.0), whereas withdrawn students logged a mean of **334.3 clicks** (median: 217.0) across **15.3 active days** (median: 13.0).
- **Effect Size:** Mann-Whitney U test confirms statistical significance ($p = 1.36 \times 10^{-33}$), Cliff's delta $= -0.1117$, Cohen's d $= -0.1948$.
- **Exploration Breadth:** Retained students accessed a median of 26 distinct resource sites compared to 24 for withdrawn students ($p = 2.47 \times 10^{-11}$, Cliff's delta $= -0.0617$).


In [ ]:
stat_df = pd.read_csv(TABLES_DIR / "statistical_comparisons.csv")
display(stat_df[stat_df["feature"].str.startswith("vle_")])

# Display Figure 2
display(Image(filename=str(FIGURES_DIR / "fig2_vle_clicks_distribution_by_outcome.png")))


## 4. Observation 3: Longitudinal Engagement Decay & Weekly Activity Curves

Examining weekly click trajectories from Week -3 through Week 5 reveals a crucial behavioral inflection:
1. **Pre-Course Invariance:** During Weeks -3 to -1, interaction levels between eventual completers and dropouts are virtually indistinguishable.
2. **Week 0 Divergence:** A noticeable gap emerges immediately upon course commencement in Week 0 and widens substantially through Weeks 2, 3, 4, and 5.
3. **Inactivity Recency:** Eventual dropouts exhibit significantly longer periods of silence by Day 35 (median 3.0 days elapsed since last click vs 2.0 days for completers, $p = 8.22 \times 10^{-36}$, Cliff's delta $= 0.1140$).


In [ ]:
weekly_df = pd.read_csv(TABLES_DIR / "weekly_activity_curves.csv")
display(weekly_df.head(10))

# Display Figure 3
display(Image(filename=str(FIGURES_DIR / "fig3_weekly_vle_activity_curves.png")))


## 5. Observation 4: Early Coursework Milestone Failure as a High-Confidence Signal

Across the 18 module-presentations with continuous assessments due on or before Day 35 ($N = 24,700$ eligible students under the Reasonable Opportunity Policy; 4 offerings have 0 assessments due $\le$ Day 35):
- **Submission Adherence:** Retained students achieved a mean submission rate of **90.7%** (with mean 0.813 on-time submissions relative to scheduled coursework), compared to only **78.4%** (mean 0.616 on-time) for eventual dropouts.
- **Odds Ratio:** Failing to submit at least one due assessment by Day 35 yields an **Odds Ratio of 2.80** (2.7976, 95% CI: [2.58, 3.03], Fisher exact $p = 1.72 \times 10^{-132}$).
- **Reasonable Opportunity & Non-Penalty Invariants:**
  - For learners in offerings with zero assessments due by Day 35 (`DDD 2013B`, `GGG 2013J`, `GGG 2014B`, `GGG 2014J`), or learners whose module assessments fell strictly prior to late registration without banked satisfaction, `submission_rate` is represented as undefined (`NaN`) and `has_missed_due_assessment` is 0.
  - Across the cohort, exactly 5 late-registering records had deadlines prior to registration without prior banked satisfaction:
    - 2 students had all due assessments scheduled prior to registration and were excluded from the assessment-due denominator ($N: 24,702 \to 24,700$; `BBB 2013J 1170618` [retained, leaves Cell b] and `FFF 2014B 628323` [withdrawn, leaves Cell a]).
    - 2 students submitted their subsequent post-registration assessment on time and were reclassified from missed to non-missed (`CCC 2014J 350267` [withdrawn, moves from Cell a to Cell c] and `CCC 2014J 245600` [retained, moves from Cell b to Cell d]).
    - 1 student remained classified as missed because their subsequent post-registration assessment was also unsubmitted (`CCC 2014J 589463` [retained, stays in Cell b]).
  - Verified contingency table: Withdrawn Missed $(a) = 1,192$, Retained Missed $(b) = 2,281$, Withdrawn Non-Missed $(c) = 3,341$, Retained Non-Missed $(d) = 17,886$.


In [ ]:
assess_df = pd.read_csv(TABLES_DIR / "assessment_summary_by_outcome.csv")
display(assess_df)

# Display Figure 5 (Assessment Submission Progress)
display(Image(filename=str(FIGURES_DIR / "fig5_assessment_submission_progress.png")))


## 6. Observation 5: Registration Pacing & Commitment Horizon

Students registering far in advance of module start exhibit significantly higher persistence ($N = 27,128$):
- **Median Registration Timing:** Retained students registered a median of **-52.0 days** before commencement, compared to **-58.0 days** for withdrawn students ($p = 1.72 \times 10^{-9}$, Cliff's delta $= -0.0557$).
- Late registration closer to or after the start date is associated with higher attrition risk.


In [ ]:
# Display Figure 4 (Registration Lead Time)
display(Image(filename=str(FIGURES_DIR / "fig4_registration_lead_time_distribution.png")))


## 7. Observation 6: Missingness and Contextual Demographic Indicators

- **Missingness Stability:** `imd_band` missingness occurs in 3.90% of retained students vs 3.09% of withdrawn students (unadjusted Fisher's exact $p = 0.0079$). While nominally statistically significant due to the large sample size ($N = 27,128$), the absolute difference of 0.81 percentage points represents a negligible practical effect size rather than substantial selective attrition bias. This unadjusted difference should not be interpreted as evidence of severe selective reporting bias.
- **Academic History:** Eventual withdrawals exhibit a slightly higher average number of previous course attempts (`num_of_prev_attempts`; retained mean: 0.15, withdrawn mean: 0.20; Mann-Whitney U $p = 7.79 \times 10^{-12}$), reflecting prior academic struggle as a background indicator.

In [ ]:
missing_df = pd.read_csv(TABLES_DIR / "missingness_comparison.csv")
display(missing_df)


## 8. Feature Correlation & Methodological Limitations

### Multicollinearity Structure
Several engagement features exhibit collinearity (e.g., `vle_total_clicks` and `vle_recent_clicks`, $r \approx 0.88$). Understanding this correlation structure informs feature selection and regularized modeling in Phase 5.

### Repeated-Observation Clustering Limitation
The 27,128 Policy C cohort observations comprise enrollments across 24,567 unique individuals (2,450 students enroll in 2 to 5 offerings, accounting for 5,011 records). Row-level statistical tests (Mann-Whitney U, Fisher's exact test) assume independent observations; mild positive intra-student correlation may slightly understate standard errors and overstate asymptotic precision. These statistics serve as exploratory heuristic ranking signals rather than formal confirmatory causal proofs. Phases 5 and 6 will enforce grouped cross-validation blocking on student identifiers to ensure independence between training and test sets.


In [ ]:
# Display Figure 6 (Feature Correlation Matrix)
display(Image(filename=str(FIGURES_DIR / "fig6_correlation_matrix_early_signals.png")))


## 9. Synthesis & Candidate Features Recommended for Phase 5

Based on the empirical evidence gathered in Phase 4, the following candidate features are recommended for construction in **Phase 5 (Temporal Feature Engineering)**:

| Category | Feature Name | Description | Justification |
| :--- | :--- | :--- | :--- |
| **VLE Volume** | `vle_total_clicks` | Total clicks in $[-25, 35]$ | Significant volume difference ($p = 1.36 \times 10^{-33}$, Cliff's $\delta = -0.1117$) |
| **VLE Consistency** | `vle_active_days` | Number of distinct active days | Pacing regularity marker ($17.7$ vs $15.3$ days) |
| **VLE Breadth** | `vle_distinct_sites` | Distinct resources accessed | Curriculum exploration breadth ($26$ vs $24$ sites) |
| **VLE Recency** | `vle_recent_clicks` | Clicks in recent window $[22, 35]$ | Captures disengagement in the 2 weeks before cutoff |
| **VLE Velocity** | `vle_click_velocity_ratio` | Recent clicks / overall click rate | Velocity drop-off indicator |
| **VLE Inactivity** | `days_since_last_vle` | Days from last click to Day 35 (sentinel 65) | Dropout recency marker ($p = 8.22 \times 10^{-36}$) |
| **Assessment Expectation** | `assessments_due_count` | Continuous assessments due $\le$ Day 35 (with opportunity) | Course pacing baseline |
| **Assessment Submission** | `submissions_count_lte_cutoff` | Submissions turned in $\le$ Day 35 | Coursework progress indicator |
| **Assessment Punctuality** | `submitted_on_time_count` | On-time submissions $\le$ Day 35 | Deadline adherence indicator |
| **Assessment Milestone** | `has_missed_due_assessment` | Indicator for 1+ missed due assessments | Strongest single risk factor (OR $= 2.80$, $p = 1.72 \times 10^{-132}$) |
| **Registration Lead Time** | `date_registration` | Days between registration and start | Early commitment proxy ($-52$ vs $-58$ days) |
| **Curriculum Context** | `presentation_season` | February vs October presentation | Controls for seasonal withdrawal disparity |
